# 15 Date and Time Functions

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Parse strings into dates and timestamps safely, do date arithmetic, extract parts, truncate to periods, format output, and handle time zones correctly with the session time zone, <code>from_utc_timestamp</code> and <code>to_utc_timestamp</code>.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Every pipeline is partitioned, filtered or aggregated by time: daily loads, monthly revenue, "orders in the last 30 days", SLA durations. Date bugs are subtle: a wrong format pattern gives errors or nulls, a time zone mix-up shifts revenue into the wrong day, and month arithmetic has edge cases. With ANSI mode on (serverless, Spark 4), a badly formatted date <b>fails the query</b> instead of quietly becoming null.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Dates, timestamps and time zones</b>
<ul><li><code>DATE</code> is a calendar day with no time and no time zone</li><li><code>TIMESTAMP</code> is an instant in time, stored internally as microseconds since 1970-01-01 <b>UTC</b>, and <b>displayed</b> in the session time zone (<code>spark.sql.session.timeZone</code>)</li><li><code>TIMESTAMP_NTZ</code> is a "wall clock" timestamp with <b>no</b> time zone (Spark 3.4+)</li><li>Parse with <code>to_date(col, fmt)</code> / <code>to_timestamp(col, fmt)</code>. Format with <code>date_format(col, fmt)</code></li><li>Patterns are case sensitive: <code>yyyy</code> year, <code>MM</code> month, <code>dd</code> day, <code>HH</code> 24-hour, <code>mm</code> <b>minutes</b>, <code>ss</code> seconds</li><li>Under ANSI mode, invalid input raises an error. <code>try_to_timestamp</code> returns null instead</li></ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A food delivery company stores order times in UTC. A Dubai restaurant's 11:30 PM order on 31 December is stored as 19:30 UTC on 31 December, but an order at 2:00 AM Dubai time on 1 January is stored as 22:00 UTC on 31 December. If the "revenue per local day" report groups by the UTC date, New Year's Day revenue is assigned to the previous year. The fix is to convert to the restaurant's local time <b>before</b> extracting the date.
</div>

## Syntax

```python
from pyspark.sql import functions as F

F.to_date("s", "dd/MM/yyyy")                 # string -> date
F.to_timestamp("s", "yyyy-MM-dd HH:mm:ss")   # string -> timestamp
F.try_to_timestamp("s", F.lit("dd/MM/yyyy")) # null instead of an error
F.date_format("ts", "yyyy-MM")               # date/timestamp -> string
F.date_add("d", 7), F.add_months("d", 1), F.datediff("end", "start")
F.year("d"), F.month("d"), F.dayofweek("d"), F.weekofyear("d")
F.date_trunc("month", "ts"), F.last_day("d")
F.from_utc_timestamp("ts", "Asia/Dubai")     # UTC -> local wall-clock time
```

| Pattern | Meaning | Example |
|---|---|---|
| `yyyy` | year | 2024 |
| `MM` / `MMM` | month number / short name | 03 / Mar |
| `dd` | day of month | 07 |
| `HH` / `hh` | hour 0-23 / 1-12 | 18 / 06 |
| `mm` | **minutes** | 45 |
| `ss` | seconds | 09 |
| `a` | AM / PM | PM |
| `E` | day name | Thu |

## Setup: sample data

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates delivery orders with timestamps stored as UTC strings, plus dates in several raw formats.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A 6-row table. The schema shows every column as <code>string</code>, which is typical for raw data.
</div>

In [0]:
from pyspark.sql import functions as F

# Make results reproducible: interpret and display timestamps in UTC for this notebook.
spark.conf.set("spark.sql.session.timeZone", "UTC")

orders = spark.createDataFrame(
    [
        (1, "2024-12-31 19:30:00", "2024-12-31 20:05:00", "31/12/2024", "Asia/Dubai"),
        (2, "2024-12-31 22:00:00", "2024-12-31 22:41:30", "01/01/2025", "Asia/Dubai"),
        (3, "2025-01-15 08:15:00", "2025-01-15 09:00:00", "15/01/2025", "Asia/Kolkata"),
        (4, "2025-02-28 23:50:00", "2025-03-01 00:20:00", "28/02/2025", "Europe/London"),
        (5, "2025-03-10 12:00:00", "2025-03-10 12:30:00", "2025-03-10", "America/New_York"),
        (6, "2025-03-31 06:45:00", None, "n/a", "Asia/Kolkata"),
    ],
    "order_id INT, ordered_utc STRING, delivered_utc STRING, business_date_raw STRING, restaurant_tz STRING",
)
orders.printSchema()
display(orders)

## 1. Parsing strings into timestamps and dates

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Converts the ISO-formatted strings with <code>to_timestamp</code>, which needs no pattern because they already follow the default <code>yyyy-MM-dd HH:mm:ss</code>.<br><br>
<b>Why it matters</b><br>Typed timestamps enable arithmetic, comparisons and time zone conversions. Strings only sort and compare as text.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>ordered_at</code> and <code>delivered_at</code> show as <code>timestamp</code> in the schema. Order 6 has a null <code>delivered_at</code>, because the input is null (nulls pass through without an error).
</div>

In [0]:
parsed = orders.withColumns({
    "ordered_at": F.to_timestamp("ordered_utc"),
    "delivered_at": F.to_timestamp("delivered_utc"),
})
parsed.select("order_id", "ordered_at", "delivered_at").printSchema()
parsed.select("order_id", "ordered_at", "delivered_at").show()

## 2. Custom formats, mixed formats and ANSI mode

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Parses <code>business_date_raw</code>, which mixes <code>dd/MM/yyyy</code>, <code>yyyy-MM-dd</code> and junk, first strictly and then safely.<br><br>
<b>Why it matters</b><br>A strict <code>to_date</code> with one pattern <b>fails the whole query</b> on the first non-matching row when ANSI mode is on. <code>try_to_timestamp</code> returns null for rows that don't match, and <code>coalesce</code> over several patterns handles mixed formats.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The strict parse prints an error (such as <code>CANNOT_PARSE_TIMESTAMP</code> or <code>CAST_INVALID_INPUT</code>). The safe version shows a date for rows 1 to 5 and null for <code>n/a</code>.
</div>

In [0]:
try:
    orders.select(F.to_date("business_date_raw", "dd/MM/yyyy")).show()
except Exception as e:
    print("Strict parse failed:", type(e).__name__, "-", str(e).splitlines()[0][:150])

In [0]:
# try_to_timestamp needs the format as a Column, so wrap it in F.lit().
safe_date = F.coalesce(
    F.try_to_timestamp("business_date_raw", F.lit("dd/MM/yyyy")),
    F.try_to_timestamp("business_date_raw", F.lit("yyyy-MM-dd")),
).cast("date")

dated = parsed.withColumn("business_date", safe_date)
dated.select("order_id", "business_date_raw", "business_date").show()

## 3. Extracting parts of dates

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Pulls out year, quarter, month, day, weekday, week number and hour.<br><br>
<b>Why it matters</b><br>These are the columns you group by in reports, and the columns you partition tables by.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Watch out</b><br>
<code>dayofweek</code> returns <b>1 for Sunday</b> through 7 for Saturday (US convention). <code>weekday</code> returns <b>0 for Monday</b> through 6 for Sunday. Mixing them up shifts weekend logic by a day.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
For order 1 (2024-12-31 19:30 UTC): year 2024, quarter 4, month 12, day 31, <code>dayofweek</code> 3 (Tuesday), <code>day_name</code> Tue, hour 19.
</div>

In [0]:
dated.select(
    "order_id", "ordered_at",
    F.year("ordered_at").alias("year"),
    F.quarter("ordered_at").alias("q"),
    F.month("ordered_at").alias("month"),
    F.dayofmonth("ordered_at").alias("day"),
    F.dayofweek("ordered_at").alias("dayofweek"),
    F.date_format("ordered_at", "E").alias("day_name"),
    F.weekofyear("ordered_at").alias("week"),
    F.hour("ordered_at").alias("hour"),
).show()

## 4. Date arithmetic and durations

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Adds days and months, computes day differences, and measures delivery time in minutes.<br><br>
<b>Why it matters</b><br>SLAs ("delivered within 45 minutes"), retention windows ("last 30 days") and billing cycles all need arithmetic. For durations between timestamps, subtract epoch seconds or use <code>timestampdiff</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>delivery_minutes</code>: 35.0, 41.5, 45.0, 30.0, 30.0 and null for order 6. <code>days_since_order</code> relative to 2025-04-01. <code>plus_1_month</code> for 2025-01-15 is 2025-02-15.
</div>

In [0]:
dated.select(
    "order_id",
    F.to_date("ordered_at").alias("order_date"),
    F.date_add(F.to_date("ordered_at"), 7).alias("plus_7_days"),
    F.add_months(F.to_date("ordered_at"), 1).alias("plus_1_month"),
    F.datediff(F.lit("2025-04-01").cast("date"), F.to_date("ordered_at")).alias("days_since_order"),
    ((F.unix_timestamp("delivered_at") - F.unix_timestamp("ordered_at")) / 60).alias("delivery_minutes"),
    F.expr("timestampdiff(MINUTE, ordered_at, delivered_at)").alias("minutes_whole"),
).show()

## 5. Edge cases: month ends and leap years

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Adds one month to 31 January, and compares <code>months_between</code> with <code>timestampdiff(MONTH, ...)</code>.<br><br>
<b>Why it matters</b><br>Month arithmetic is not simple day arithmetic. <code>add_months</code> clamps to the last valid day. <code>months_between</code> treats two month-end dates as a whole number of months, while <code>timestampdiff</code> counts completed calendar months. Billing and tenure calculations depend on choosing the right one.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>2024-01-31 + 1 month = 2024-02-29</code> (leap year) and <code>2025-01-31 + 1 month = 2025-02-28</code>. <code>months_between('2024-02-29', '2024-01-31')</code> is <code>1.0</code>, but <code>timestampdiff(MONTH, ...)</code> is <code>0</code>.
</div>

In [0]:
spark.sql("""
    SELECT add_months(date'2024-01-31', 1)                          AS leap_year,
           add_months(date'2025-01-31', 1)                          AS normal_year,
           last_day(date'2024-02-10')                               AS last_day_feb_2024,
           months_between(date'2024-02-29', date'2024-01-31')       AS months_between,
           timestampdiff(MONTH, date'2024-01-31', date'2024-02-29') AS timestampdiff_months
""").show()

## 6. Truncating and formatting for reports

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Truncates timestamps to month, week and hour with <code>date_trunc</code> / <code>trunc</code>, and formats them as strings with <code>date_format</code>.<br><br>
<b>Why it matters</b><br><code>date_trunc</code> keeps a real timestamp, which sorts and joins correctly. <code>date_format</code> produces text for display. Grouping by <code>date_trunc("month", ts)</code> is the standard way to build monthly reports.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>month_start</code> is the first instant of each month, <code>week_start</code> is a Monday, <code>pretty</code> looks like <code>31 Dec 2024, 07:30 PM</code>. The final table counts orders per month.
</div>

In [0]:
dated.select(
    "ordered_at",
    F.date_trunc("month", "ordered_at").alias("month_start"),
    F.date_trunc("week", "ordered_at").alias("week_start"),
    F.date_trunc("hour", "ordered_at").alias("hour_start"),
    F.trunc(F.to_date("ordered_at"), "quarter").alias("quarter_start_date"),
    F.date_format("ordered_at", "dd MMM yyyy, hh:mm a").alias("pretty"),
).show(truncate=False)

dated.groupBy(F.date_trunc("month", "ordered_at").alias("month")).count().orderBy("month").show()

## 7. Time zones: UTC storage, local reporting

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Converts UTC timestamps to each restaurant's local time with <code>from_utc_timestamp</code>, then compares the UTC date with the local date.<br><br>
<b>Why it matters</b><br>This is the real-world example from the top. Store timestamps in UTC, and convert to local time only to compute local business dates or to display. Use region IDs like <code>Asia/Dubai</code>, not fixed offsets like <code>+04:00</code>, so daylight saving time is handled for zones that use it (London, New York).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Order 2 is on <b>2024-12-31</b> in UTC but <b>2025-01-01</b> in Dubai (22:00 UTC + 4 hours = 02:00). Order 4 stays on 28 February in London (winter time, UTC+0).
</div>

In [0]:
local = dated.withColumn("ordered_local", F.from_utc_timestamp("ordered_at", F.col("restaurant_tz")))

local.select(
    "order_id", "restaurant_tz", "ordered_at",
    F.to_date("ordered_at").alias("utc_date"),
    "ordered_local",
    F.to_date("ordered_local").alias("local_date"),
).show(truncate=False)

## 8. How the session time zone changes what you see

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Displays the same timestamp value with the session time zone set to UTC and then to Asia/Kolkata, and shows that the stored epoch value doesn't change.<br><br>
<b>Why it matters</b><br>A <code>TIMESTAMP</code> is an instant. Changing <code>spark.sql.session.timeZone</code> changes how it is <b>displayed</b> and how strings <b>without</b> an offset are <b>parsed</b>, not the instant itself. Teams should set a single session time zone (usually UTC) for all pipelines.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
<code>spark.sql.session.timeZone</code> is one of the settings you <b>can</b> change on serverless compute. If your workspace rejects it, read the default with <code>spark.conf.get</code> and note it in the troubleshooting log.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
First two tables: the <b>same</b> <code>epoch_seconds</code> (1736928900), but <code>ts</code> displays as 08:15 in UTC and 13:45 in Asia/Kolkata. Last table: the same <b>string</b> parsed under the two time zones gives two <b>different</b> epoch values, 19,800 seconds (5.5 hours) apart.
</div>

In [0]:
# A fixed instant: 1736928900 seconds after the epoch = 2025-01-15 08:15:00 UTC.
instant = spark.sql("SELECT timestamp_seconds(1736928900) AS ts")

for tz in ["UTC", "Asia/Kolkata"]:
    spark.conf.set("spark.sql.session.timeZone", tz)
    instant.select(F.lit(tz).alias("session_tz"), "ts", F.unix_timestamp("ts").alias("epoch_seconds")).show()

# Parsing a string WITHOUT an offset uses the session time zone, so the instant changes.
results = []
for tz in ["UTC", "Asia/Kolkata"]:
    spark.conf.set("spark.sql.session.timeZone", tz)
    epoch = spark.sql("SELECT unix_timestamp(to_timestamp('2025-01-15 08:15:00')) AS e").first()["e"]
    results.append((tz, epoch))
spark.createDataFrame(results, "session_tz STRING, parsed_epoch BIGINT").show()

# Always set it back so the rest of the notebook behaves the same.
spark.conf.set("spark.sql.session.timeZone", "UTC")

## 9. Building a date dimension

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Generates one row per day for a year with <code>sequence</code> and <code>explode</code>, and adds calendar attributes.<br><br>
<b>Why it matters</b><br>Almost every Gold layer has a <code>dim_date</code> table. Generating it in Spark is a short, classic exercise, and filling gaps in time series ("days with zero orders") needs it too.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
365 rows for 2025, the first row being 2025-01-01 (a Wednesday), with <code>is_weekend</code>, <code>month_name</code> and <code>quarter</code> columns.
</div>

In [0]:
dim_date = (
    spark.sql("SELECT explode(sequence(date'2025-01-01', date'2025-12-31', interval 1 day)) AS date")
    .withColumns({
        "year": F.year("date"),
        "quarter": F.quarter("date"),
        "month": F.month("date"),
        "month_name": F.date_format("date", "MMMM"),
        "day_name": F.date_format("date", "EEEE"),
        "is_weekend": F.dayofweek("date").isin(1, 7),
    })
)
print("Rows:", dim_date.count())
dim_date.show(5)

## Under the hood

```
"2024-12-31 22:00:00"  --to_timestamp-->  1735682400000000 µs since epoch (UTC)
                                                |
                    display in session TZ  <----+----> from_utc_timestamp(ts, 'Asia/Dubai')
                    (UTC: 2024-12-31 22:00)            shifts the value so it DISPLAYS as Dubai wall-clock time
                                                       (2025-01-01 02:00)
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Narrow</span> date functions are row-level expressions with no shuffle.

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> `to_date`, `date_trunc` and the rest are Catalyst expressions, so they are optimized and code-generated.

Filters on date columns are pushed down to Delta and Parquet scans, where file statistics (min/max per column) let Spark skip files. A filter like `order_date >= '2025-06-01'` can be pushed down and compared to those statistics. A filter that wraps the column in a function, like `year(order_date) = 2025`, usually **can't** be pushed down, so every file is read. (Filters on **partition** columns are an exception: Spark evaluates them against the partition values, so they still prune.)

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
In the first plan, the scan's <code>PushedFilters</code> (or <code>DataFilters</code> on Databricks) include the comparison on <code>event_date</code>. In the second, only <code>IsNotNull(event_date)</code> is pushed, and the <code>year(...)</code> filter runs after the scan.
</div>

In [0]:
events = spark.range(1000).withColumn("event_date", F.expr("date_add(date'2025-01-01', cast(id % 365 AS INT))"))
events.write.mode("overwrite").saveAsTable("lesson15_events")

print("---- range filter on the raw column ----")
spark.table("lesson15_events").filter(F.col("event_date") >= "2025-06-01").explain()

print("---- function wrapped around the column ----")
spark.table("lesson15_events").filter(F.year("event_date") == 2025).explain()

spark.sql("DROP TABLE IF EXISTS lesson15_events")

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>CANNOT_PARSE_TIMESTAMP</code> / <code>CAST_INVALID_INPUT</code> on dates</b><br>
ANSI mode is on and a value doesn't match the pattern. Use <code>try_to_timestamp</code> with <code>coalesce</code> over the formats you expect, and count the nulls.<br><br>
<b>⛔ Problem: minutes come out as months (or the other way round)</b><br><code>MM</code> is month and <code>mm</code> is minutes. <code>HH</code> is 0-23 and <code>hh</code> is 1-12 (needs <code>a</code> for AM/PM).<br><br>
<b>⛔ Problem: revenue lands on the wrong day</b><br>Dates were extracted in UTC instead of local time. Convert with <code>from_utc_timestamp(ts, region_id)</code> first.<br><br>
<b>⛔ Problem: results differ between notebooks or jobs</b><br>Different session time zones. Set <code>spark.sql.session.timeZone</code> explicitly.<br><br>
<b>⛔ Problem: weekend flag is off by one day</b><br><code>dayofweek</code> is 1 = Sunday, while <code>weekday</code> is 0 = Monday.<br><br>
<b>⛔ Problem: date filter reads every file</b><br>The filter wraps the column in a function (<code>year(d) = 2025</code>), so it can't be pushed down for data skipping. Filter on a range of the raw column instead.<br><br>
**⛔ Problem: <code>Fail to recognize pattern</code> errors**<br>Some legacy patterns (for example a week-based year <code>YYYY</code> combined with <code>MM</code>) are rejected by Spark 3+. Use <code>yyyy</code>.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is the difference between <code>DATE</code>, <code>TIMESTAMP</code> and <code>TIMESTAMP_NTZ</code>?</b><br>
<code>DATE</code> is a calendar day. <code>TIMESTAMP</code> is an instant, stored as microseconds since the epoch in UTC and displayed in the session time zone. <code>TIMESTAMP_NTZ</code> is a wall-clock value with no time zone that is never converted.<br><br>

<b>🎤 2. How do you handle dates that arrive in several formats?</b><br>
Parse each format with <code>try_to_timestamp</code> and take the first success with <code>coalesce</code>, then cast to date. Count the rows that are still null and route them to a quarantine table.<br><br>

<b>🎤 3. What does the session time zone affect?</b><br>
How timestamps are displayed, and how strings without an explicit offset are parsed into timestamps. It doesn't change the stored instant. Pipelines should set it explicitly, usually to UTC.<br><br>

<b>🎤 4. How do you compute the local business date for global data?</b><br>
Keep the timestamp in UTC, convert with <code>from_utc_timestamp(ts, 'Region/City')</code> and then take <code>to_date</code>. Use region IDs so daylight saving time is handled.<br><br>

<b>🎤 5. What is the difference between <code>date_trunc</code> and <code>date_format</code>?</b><br>
<code>date_trunc</code> returns a timestamp rounded down to a unit such as a month, which is good for grouping and joins. <code>date_format</code> returns a string for display.<br><br>

<b>🎤 6. How do you calculate the duration between two timestamps in minutes?</b><br>
<code>(unix_timestamp(end) - unix_timestamp(start)) / 60</code>, or <code>timestampdiff(MINUTE, start, end)</code> for whole minutes.<br><br>

<b>🎤 7. Why can <code>WHERE year(order_date) = 2025</code> be slower than <code>WHERE order_date &gt;= '2025-01-01' AND order_date &lt; '2026-01-01'</code>?</b><br>
The range filter on the raw column can be pushed down to the scan and compared with file min/max statistics, so files outside the range are skipped. The function version usually can't be pushed down, so every file is read and filtered afterwards. Filters on partition columns are the exception, because they are evaluated against partition values.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. Column <code>ts</code> holds UTC timestamps. Which expression returns the calendar date in Singapore?</b><br>
A. <code>to_date(ts, 'Asia/Singapore')</code><br>
B. <code>to_date(from_utc_timestamp(ts, 'Asia/Singapore'))</code><br>
C. <code>to_date(to_utc_timestamp(ts, 'Asia/Singapore'))</code><br>
D. <code>date_format(ts, 'Asia/Singapore')</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>from_utc_timestamp</code> shifts a UTC instant to the local wall-clock time, and then <code>to_date</code> takes the local date. The second argument of <code>to_date</code> is a format pattern, not a time zone.</details><br><br>

<b>Q2. Which format pattern parses <code>'2025-03-07 18:45:09'</code>?</b><br>
A. <code>yyyy-mm-dd HH:MM:ss</code><br>
B. <code>yyyy-MM-dd HH:mm:ss</code><br>
C. <code>YYYY-MM-DD hh:mm:ss</code><br>
D. <code>yyyy-MM-dd hh:MM:SS</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>MM</code> is month, <code>mm</code> is minutes, <code>HH</code> is 24-hour, <code>ss</code> is seconds, and <code>dd</code> is day of month.</details><br><br>

<b>Q3. What is returned by <code>add_months(DATE'2025-01-31', 1)</code>?</b><br>
A. <code>2025-03-03</code><br>
B. <code>2025-02-28</code><br>
C. <code>2025-02-31</code><br>
D. NULL<br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>add_months</code> clamps to the last day of the target month.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b><br>
Using <code>dated</code> (or rebuild it by running the cells above):
<ol><li>Add <code>order_hour_local</code>: the hour of the order in the restaurant's local time</li><li>Add <code>is_late</code>: true when delivery took more than 40 minutes</li><li>Count orders per local date (not UTC date)</li><li>Parse these values into dates in one expression, returning null for anything unparseable: <code>"2025-03-01"</code>, <code>"01/03/2025"</code>, <code>"March 1, 2025"</code>, <code>"tomorrow"</code></li><li>Build a table of every month start in 2025 with the number of days in that month (<code>last_day</code>)</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1-2. Local hour and late flag
sol = dated.withColumns({
    "order_hour_local": F.hour(F.from_utc_timestamp("ordered_at", F.col("restaurant_tz"))),
    "is_late": (F.unix_timestamp("delivered_at") - F.unix_timestamp("ordered_at")) > 40 * 60,
})
sol.select("order_id", "restaurant_tz", "order_hour_local", "is_late").show()

# 3. Orders per LOCAL date
sol.groupBy(F.to_date(F.from_utc_timestamp("ordered_at", F.col("restaurant_tz"))).alias("local_date")) \
   .count().orderBy("local_date").show()

# 4. Several formats in one expression
raw_dates = spark.createDataFrame([("2025-03-01",), ("01/03/2025",), ("March 1, 2025",), ("tomorrow",)], "s STRING")
raw_dates.select(
    "s",
    F.coalesce(
        F.try_to_timestamp("s", F.lit("yyyy-MM-dd")),
        F.try_to_timestamp("s", F.lit("dd/MM/yyyy")),
        F.try_to_timestamp("s", F.lit("MMMM d, yyyy")),
    ).cast("date").alias("parsed"),
).show()

# 5. Month starts and days per month
spark.sql("SELECT explode(sequence(date'2025-01-01', date'2025-12-01', interval 1 month)) AS month_start") \
    .withColumn("days_in_month", F.dayofmonth(F.last_day("month_start"))).show()

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Parse with <code>to_date</code> / <code>to_timestamp</code>. Under ANSI mode bad input <b>fails</b>, so use <code>try_to_timestamp</code> + <code>coalesce</code> for messy data</li><li>Pattern letters are case sensitive: <code>MM</code> month, <code>mm</code> minutes, <code>HH</code> 24-hour</li><li><code>TIMESTAMP</code> = instant in UTC, displayed in the session time zone. Set the session time zone explicitly</li><li>Store UTC, convert with <code>from_utc_timestamp(ts, 'Region/City')</code> for local dates</li><li><code>date_add</code>, <code>add_months</code> (clamps month ends), <code>datediff</code>, <code>months_between</code>, <code>timestampdiff</code></li><li><code>date_trunc</code> for grouping, <code>date_format</code> for display</li><li><code>dayofweek</code>: 1 = Sunday. <code>weekday</code>: 0 = Monday</li><li>Filter on raw date ranges so the filter is pushed down for data skipping</li></ul>
</div>

| Task | Code |
|---|---|
| Parse | `F.to_timestamp("s", "dd/MM/yyyy HH:mm")` |
| Safe parse | `F.try_to_timestamp("s", F.lit("dd/MM/yyyy"))` |
| Format | `F.date_format("ts", "yyyy-MM")` |
| Add | `F.date_add("d", 7)`, `F.add_months("d", 1)` |
| Difference | `F.datediff("a", "b")`, `timestampdiff(MINUTE, a, b)` |
| Truncate | `F.date_trunc("month", "ts")` |
| Time zone | `F.from_utc_timestamp("ts", "Asia/Dubai")` |
| Date range | `explode(sequence(start, end, interval 1 day))` |

## Git commit

```
git add 01_spark_basics/15_date_functions.ipynb
git commit -m "add 15 date functions notebook"
```